# Dravet syndrome knowledge graph - the SCN1A brain interactome

Dravet syndrome (severe myoclonic epilepsy of infancy) is a developmental and
epileptic encephalopathy caused in >80% of patients by loss-of-function variants
in **SCN1A**, the gene encoding the Na<sub>v</sub>1.1 sodium-channel alpha subunit.

This pipeline builds an annotated, brain-specific molecular network centred on
SCN1A:

1. **Section 1-2** - assemble a high confidence human interactome from seven physical-PPI
   resources (STRING, BioGRID, IntAct, Hi-Union, Lit-BM, SIGNOR, APID) and merge
   them into one canonical undirected edge table.
2. **Section 3** - restrict it to brain-expressed proteins (Human Protein Atlas
   brain-region RNA).
3. **Section 4** - extract the SCN1A neighbourhood: SCN1A, its interactors, and
   the interactors of those interactors (`MAX_HOPS`), keeping every edge among
   them.

In [ ]:
from pypath import omnipath
from pathlib import Path
import pandas as pd
import networkx as nx
from pypath.utils import mapping
import urllib.request

omnipath.db

DB_DIR = Path("dbs") # working directory with downloaded datasets
data_dir = Path("data") # folder where the manually downloaded data sit
DB_DIR.mkdir(exist_ok=True)

[2026-08-05 12:08:54] [DEBUG] [pkg_infra.session        ] ▸ Using app logger: default
[2026-08-05 12:08:54] [DEBUG] [pkg_infra.session        ] ▸ Session time initialized: utc=2026-08-05 10:08:54.114061+00:00, local=2026-08-05 12:08:54.114063+02:00, timezone=CEST
[2026-08-05 12:08:54] [DEBUG] [pkg_infra.session        ] ▸ Location lookup skipped
[2026-08-05 12:08:54] [INFO ] [pkg_infra.session        ] ▸ Creating new session...
[2026-08-05 12:08:54] [INFO ] [pkg_infra.session        ] ▸ Session has been created
[2026-08-05 12:08:54] [WARNING] [dlmachine._metadata      ] ▸ No local pyproject metadata found, trying installed metadata
[2026-08-05 12:08:54] [DEBUG] [pkg_infra.session        ] ▸ get_session called with workspace=., include_location=False, config_path=None
[2026-08-05 12:08:54] [INFO ] [pkg_infra.session        ] ▸ Initialization of a session:
[2026-08-05 12:08:54] [INFO ] [pkg_infra.session        ] ▸ Reusing existing session
[2026-08-05 12:08:54] [INFO ] [pkg_infra.session

### String

In [2]:
from pypath.inputs import string

string_mapped_csv = DB_DIR / "string_mapped.csv"

if string_mapped_csv.exists():
    df_string = pd.read_csv(string_mapped_csv)
    print(f"STRING loaded from cache ({string_mapped_csv}): {len(df_string):,} rows")
else:
    string_int = string.string_physical_interactions(
        ncbi_tax_id=9606,
        score_threshold="high_confidence",
    )

    df_string = pd.DataFrame.from_records(
        string_int,
        columns=string_int[0]._fields,
    )

    df_string.to_csv(DB_DIR / "string.csv", index=False)

    unique_ensp = pd.unique(df_string[["protein_a", "protein_b"]].values.ravel())

    ensp_to_hgnc = {
        x: next(iter(mapping.map_name(x, "ensp", "genesymbol")), None)
        for x in unique_ensp
    }

    df_string = df_string[df_string["experimental"] >= 700]

    df_string["source"] = df_string["protein_a"].map(ensp_to_hgnc)
    df_string["target"] = df_string["protein_b"].map(ensp_to_hgnc)

    df_string["database"] = "STRING"

    df_string.to_csv(string_mapped_csv, index=False)
    print(f"STRING saved: {len(df_string):,} rows")


STRING loaded from cache (dbs/string_mapped.csv): 88,980 rows


In [3]:
df_string

,protein_a,protein_b,experimental,database,textmining,combined_score,source,target
0,ENSP00000000412,ENSP00000438085,960,STRING,291,970,M6PR,GGA3
1,ENSP00000001008,ENSP00000354558,900,STRING,373,934,FKBP4,MTOR
2,ENSP00000003084,ENSP00000361930,901,STRING,93,907,CFTR,YWHAB
3,ENSP00000003084,ENSP00000358866,900,STRING,461,944,CFTR,FLNA
4,ENSP00000003084,ENSP00000306330,901,STRING,0,901,CFTR,YWHAG
...,...,...,...,...,...,...,...,...
88975,ENSP00000501277,ENSP00000294339,886,STRING,983,999,LDB1,TAL1
88976,ENSP00000501277,ENSP00000290759,712,STRING,95,728,LDB1,ISL2
88977,ENSP00000501277,ENSP00000483921,964,STRING,541,983,LDB1,SSBP2
88978,ENSP00000501317,ENSP00000305071,960,STRING,87,962,RFX7,RFXANK


### Biogrid

In [4]:
from pypath.inputs import biogrid

biogrid_csv = DB_DIR / "biogrid.csv"

if biogrid_csv.exists():
    df_biogrid = pd.read_csv(biogrid_csv)
    print(f"BioGRID loaded from cache ({biogrid_csv}): {len(df_biogrid):,} rows")
else:
    biogrid_int = biogrid.biogrid_all_interactions(
        organism=9606,
        htp_limit=20,
    )

    df_biogrid = pd.DataFrame.from_records(
        biogrid_int,
        columns=biogrid_int[0]._fields,
    )

    df_biogrid = df_biogrid.rename(columns={"partner_a": "source", "partner_b": "target"})

    exclude_types = [
        "Phenotypic Enhancement",
        "Phenotypic Suppression",
        "Synthetic Rescue",
        "Synthetic Lethality",
        "Synthetic Growth Defect",
        "Negative Genetic",
        "Positive Genetic",
        "Dosage Rescue",
        "Dosage Lethality",
        "Dosage Growth Defect",
        "Co-fractionation",
        "Co-purification",
        "Co-localization",
        "Affinity Capture-MS",
        "Affinity Capture-RNA",
        "Protein-RNA",
        "Proximity Label-MS"
    ]

    df_biogrid = df_biogrid[~df_biogrid["experimental_system"].isin(exclude_types)]

    df_biogrid["database"] = "BioGRID"
    df_biogrid.to_csv(biogrid_csv, index=False)
    print(f"BioGRID saved: {len(df_biogrid):,} rows")


BioGRID loaded from cache (dbs/biogrid.csv): 125,832 rows


### IntAct

In [5]:
from pypath.inputs import intact

intact_mapped_csv = DB_DIR / "intact_mapped.csv"

if intact_mapped_csv.exists():
    df_intact = pd.read_csv(intact_mapped_csv)
    print(f"IntAct loaded from cache ({intact_mapped_csv}): {len(df_intact):,} rows")
else:
    intact_int = intact.intact_interactions(
        organism=9606,
        only_proteins=True,
    )

    df_intact = pd.DataFrame.from_records(
        intact_int,
        columns=intact_int[0]._fields,
    )

    df_intact["pubmeds"] = df_intact["pubmeds"].apply(
        lambda x: ";".join(sorted(x)) if isinstance(x, set) else x
    )

    df_intact["methods"] = df_intact["methods"].apply(
        lambda x: ";".join(sorted(x)) if isinstance(x, set) else x
    )

    df_intact["interaction_types"] = df_intact["interaction_types"].apply(
        lambda x: ";".join(sorted(x)) if isinstance(x, set) else x
    )

    df_intact.to_csv(DB_DIR / "intact.csv", index=False)

    unique_uniprot = pd.unique(df_intact[["id_a", "id_b"]].values.ravel())

    uniprot_to_hgnc = {
        uid: next(iter(mapping.map_name(uid, "uniprot", "genesymbol")), None)
        for uid in unique_uniprot
    }

    df_intact["source"] = df_intact["id_a"].map(uniprot_to_hgnc)
    df_intact["target"] = df_intact["id_b"].map(uniprot_to_hgnc)

    exclude_types = [
        "association",
        "physical association",
        "colocalization",
        "proximity",
        "putative self interaction"
    ]

    df_intact = df_intact[~df_intact["interaction_types"].isin(exclude_types)]

    df_intact["database"] = "IntAct"
    df_intact.to_csv(intact_mapped_csv, index=False)
    print(f"IntAct saved: {len(df_intact):,} rows")


IntAct loaded from cache (dbs/intact_mapped.csv): 10,724 rows


### Hi-Union - from data directory

In [6]:
hiunion_mapped_csv = DB_DIR / "hiunion_mapped.csv"

if hiunion_mapped_csv.exists():
    huri_union = pd.read_csv(hiunion_mapped_csv)
    print(f"Hi-Union loaded from cache ({hiunion_mapped_csv}): {len(huri_union):,} rows")
else:
    hiunion_path = data_dir / "hiunion.tsv"
    hiunion_url = "https://interactome-atlas.org/data/HI-union.tsv"

    if hiunion_path.exists():
        print(f"Reading existing Hi-Union file from {hiunion_path}")
    else:
        print(f"Hi-Union file not found, downloading from {hiunion_url}")
        data_dir.mkdir(exist_ok=True)
        urllib.request.urlretrieve(hiunion_url, hiunion_path)

    huri_union = pd.read_csv(hiunion_path, sep="\t", header=None)
    huri_union.columns = ["protein_a", "protein_b"]
    huri_union.to_csv(DB_DIR / "hiunion.csv", index=False)

    unique_ensp = pd.unique(huri_union[["protein_a", "protein_b"]].values.ravel())

    ens_to_hgnc = {
        x: next(iter(mapping.map_name(x, "ensembl", "genesymbol")), None)
        for x in unique_ensp
    }

    huri_union["source"] = huri_union["protein_a"].map(ens_to_hgnc)
    huri_union["target"] = huri_union["protein_b"].map(ens_to_hgnc)

    huri_union["database"] = "Hi-Union"
    huri_union.to_csv(hiunion_mapped_csv, index=False)
    print(f"Hi-Union saved: {len(huri_union):,} rows")

huri_union


Hi-Union loaded from cache (dbs/hiunion_mapped.csv): 64,006 rows


,protein_a,protein_b,source,target,database
0,ENSG00000000005,ENSG00000061656,TNMD,SPAG4,Hi-Union
1,ENSG00000000005,ENSG00000099968,TNMD,BCL2L13,Hi-Union
2,ENSG00000000005,ENSG00000104765,TNMD,BNIP3L,Hi-Union
3,ENSG00000000005,ENSG00000105383,TNMD,CD33,Hi-Union
4,ENSG00000000005,ENSG00000114455,TNMD,HHLA2,Hi-Union
...,...,...,...,...,...
64001,ENSG00000275774,ENSG00000275774,HNRNPCL2,HNRNPCL2,Hi-Union
64002,ENSG00000276070,ENSG00000278619,CCL4L1,MRM1,Hi-Union
64003,ENSG00000276076,ENSG00000276076,NaN,NaN,Hi-Union
64004,ENSG00000276644,ENSG00000276644,DACH1,DACH1,Hi-Union


### Lit BM


In [7]:
litbm_mapped_csv = DB_DIR / "litbm_mapped.csv"

if litbm_mapped_csv.exists():
    litbm = pd.read_csv(litbm_mapped_csv)
    print(f"Lit-BM loaded from cache ({litbm_mapped_csv}): {len(litbm):,} rows")
else:
    litbm_path = data_dir / "Lit-BM.tsv"
    litbm_url = "https://interactome-atlas.org/data/Lit-BM.tsv"

    if litbm_path.exists():
        print(f"Reading existing Lit-BM file from {litbm_path}")
    else:
        print(f"Lit-BM file not found, downloading from {litbm_url}")
        data_dir.mkdir(exist_ok=True)
        urllib.request.urlretrieve(litbm_url, litbm_path)

    litbm = pd.read_csv(litbm_path, sep="\t", header=None)
    litbm.columns = ["protein_a", "protein_b"]
    litbm.to_csv(DB_DIR / "litbm.csv", index=False)

    unique_ensp = pd.unique(litbm[["protein_a", "protein_b"]].values.ravel())

    ens_to_hgnc = {
        x: next(iter(mapping.map_name(x, "ensembl", "genesymbol")), None)
        for x in unique_ensp
    }

    litbm["source"] = litbm["protein_a"].map(ens_to_hgnc)
    litbm["target"] = litbm["protein_b"].map(ens_to_hgnc)

    litbm["database"] = "Lit-BM"
    litbm.to_csv(litbm_mapped_csv, index=False)
    print(f"Lit-BM saved: {len(litbm):,} rows")

litbm


Lit-BM loaded from cache (dbs/litbm_mapped.csv): 13,442 rows


,protein_a,protein_b,source,target,database
0,protein_a,protein_b,NaN,NaN,Lit-BM
1,ENSG00000001167,ENSG00000066136,NFYA,NFYC,Lit-BM
2,ENSG00000001167,ENSG00000120837,NFYA,NFYB,Lit-BM
3,ENSG00000002016,ENSG00000002016,RAD52,RAD52,Lit-BM
4,ENSG00000002330,ENSG00000129473,BAD,BCL2L2,Lit-BM
...,...,...,...,...,...
13437,ENSG00000105185,ENSG00000172977,PDCD5,KAT5,Lit-BM
13438,ENSG00000106211,ENSG00000130402,HSPB1,ACTN4,Lit-BM
13439,ENSG00000106211,ENSG00000178028,HSPB1,DMAP1,Lit-BM
13440,ENSG00000142208,ENSG00000150907,AKT1,FOXO1,Lit-BM


### SIGNOR

SIGNOR provides directed, signed interactions (activation / inhibition / ...).  

In [8]:
from pypath.inputs import signor as signor_input

signor_mapped_csv = DB_DIR / "signor_mapped.csv"

if signor_mapped_csv.exists():
    df_signor = pd.read_csv(signor_mapped_csv)
    print(f"SIGNOR loaded from cache ({signor_mapped_csv}): {len(df_signor):,} rows")
else:
    signor_int = signor_input.signor_interactions()
    df_signor = pd.DataFrame.from_records(
        signor_int, columns=signor_int[0]._fields
    )

    df_signor = df_signor[(df_signor["source_type"] == "protein") &
    (df_signor["target_type"] == "protein")]

    df_signor.to_csv(DB_DIR / "signor.csv", index=False)

    all_ids = pd.unique(df_signor[["source", "target"]].values.ravel())
    id_map = {
        uid: next(iter(mapping.map_name(uid, "uniprot", "genesymbol")), None)
        for uid in all_ids
        if isinstance(uid, str)
    }
    df_signor["source"] = df_signor["source"].map(id_map)
    df_signor["target"] = df_signor["target"].map(id_map)
    df_signor["database"] = "SIGNOR"
    df_signor.to_csv(signor_mapped_csv, index=False)
    print(f"SIGNOR saved: {len(df_signor):,} rows")


print(f"Protein-protein interactions: {len(df_signor):,} rows")

print("SIGNOR columns:", df_signor.columns.tolist())
print("Effect values:", df_signor["effect"].value_counts().to_dict()
      if "effect" in df_signor.columns else "no effect column")


SIGNOR loaded from cache (dbs/signor_mapped.csv): 26,450 rows
Protein-protein interactions: 26,450 rows
SIGNOR columns: ['source', 'target', 'source_isoform', 'target_isoform', 'source_type', 'target_type', 'effect', 'mechanism', 'ncbi_tax_id', 'pubmeds', 'direct', 'ptm_type', 'ptm_residue', 'ptm_motif', 'database']
Effect values: {'up-regulates activity': 8185, 'up-regulates': 4690, 'down-regulates activity': 3835, 'down-regulates': 2509, 'up-regulates quantity by expression': 2357, 'down-regulates quantity by destabilization': 1615, 'down-regulates quantity by repression': 987, 'unknown': 983, 'up-regulates quantity by stabilization': 548, 'up-regulates quantity': 397, 'down-regulates quantity': 342, 'form complex': 2}


### APID

- Apid level 2 without interspecies interactions was manually downloaded from http://apid.dep.usal.es/APID/init.action in /data folder

In [9]:
apid_csv = DB_DIR / "apid_level2.csv"

if apid_csv.exists():
    df_apid = pd.read_csv(apid_csv)
    print(f"APID loaded from cache ({apid_csv}): {len(df_apid):,} rows")
else:
    apid_path = Path(data_dir) / "9606_noISI_Q1.txt"
    df_apid = pd.read_csv(apid_path, sep="\t")
    print(f"APID raw: {len(df_apid):,} rows, cols: {df_apid.columns.tolist()}")

    df_apid = df_apid.dropna(subset=["GeneName_A", "GeneName_B"]).copy()
    df_apid = df_apid.rename(columns={"GeneName_A": "source", "GeneName_B": "target"})
    df_apid["database"] = "APID"
    df_apid.to_csv(apid_csv, index=False)
    print(f"APID after filtering: {len(df_apid):,} rows")

print(f"Unique genes: {pd.unique(df_apid[['source','target']].values.ravel()).shape[0]:,}")


APID loaded from cache (dbs/apid_level2.csv): 134,022 rows
Unique genes: 15,496


## Section 2 - Merge all PPI databases

**Strategy:**
1. Normalise every edge to a canonical undirected pair `(geneA, geneB)` where `geneA ≤ geneB` alphabetically - this collapses A→B and B→A from different databases into the same record.
2. Aggregate the `databases` column as a semicolon-separated list.
3. The resulting `merged_ppi.csv` is the master edge table used for all downstream analyses.

In [10]:
def load_db(path, source_col="source", target_col="target", db_name=None,
            extra_filter=None):
    df = pd.read_csv(path)
    df = df.rename(columns={source_col: "source", target_col: "target"})
    if extra_filter is not None:
        df = df[extra_filter(df)]
    df = df.dropna(subset=["source", "target"])
    df = df[df["source"] != df["target"]]   # remove self-loops
    df["source"] = df["source"].astype(str).str.strip()
    df["target"] = df["target"].astype(str).str.strip()
    if db_name:
        df["database"] = db_name
    return df[["source", "target", "database"]].drop_duplicates()

# STRING
df_string = load_db(DB_DIR / "string_mapped.csv", db_name="STRING")

# BioGRID 
df_biogrid = load_db(DB_DIR / "biogrid.csv", db_name="BioGRID")

# IntAct
df_intact = load_db(DB_DIR / "intact_mapped.csv", db_name="IntAct")

# Hi-Union
df_hiunion = load_db(DB_DIR / "hiunion_mapped.csv", db_name="Hi-Union")

# APID
df_apid2 = load_db(DB_DIR / "apid_level2.csv", db_name="APID")

#LitBM
df_litbm = load_db(DB_DIR / "litbm_mapped.csv", db_name="Lit-BM")

#Signor
df_signor = load_db(DB_DIR / "signor_mapped.csv", db_name="SIGNOR")


for name, df in [("STRING", df_string), ("BioGRID", df_biogrid), ("IntAct", df_intact),
                  ("Hi-Union", df_hiunion), ("APID", df_apid2), ("SIGNOR", df_signor), ("Lit-BM", df_litbm)]:
    print(f"  {name:10s}: {len(df):>8,} edges")


  STRING    :   87,468 edges
  BioGRID   :   74,252 edges
  IntAct    :    4,217 edges
  Hi-Union  :   61,406 edges
  APID      :  131,130 edges
  SIGNOR    :   18,709 edges
  Lit-BM    :   12,096 edges


In [11]:
# Merge into canonical undirected edge table
import numpy as np

all_dfs = pd.concat([df_string, df_biogrid, df_intact, df_hiunion, df_apid2, df_signor, df_litbm], ignore_index=True)

all_dfs["gene_a"] = np.where(
    all_dfs["source"] <= all_dfs["target"],
    all_dfs["source"], all_dfs["target"]
)
all_dfs["gene_b"] = np.where(
    all_dfs["source"] <= all_dfs["target"],
    all_dfs["target"], all_dfs["source"]
)

# Aggregate databases per canonical edge
merged = (
    all_dfs
    .groupby(["gene_a", "gene_b"], sort=False)["database"]
    .agg(lambda x: ";".join(sorted(set(x))))
    .reset_index()
    .rename(columns={"database": "databases"})
)
merged["n_databases"] = merged["databases"].str.count(";") + 1

print(f"Total canonical edges after merge: {len(merged):,}")
print(f"Edge support distribution:")
print(merged["n_databases"].value_counts().sort_index().to_string())


Total canonical edges after merge: 232,221
Edge support distribution:
n_databases
1    155320
2     66972
3      6452
4      2368
5       830
6       252
7        27


In [12]:
merged.to_csv(DB_DIR / "merged_ppi.csv", index=False)
print(f"Saved merged_ppi.csv  - {len(merged):,} edges, {merged.columns.tolist()}")
print(f"\nUnique genes in merged network: "
      f"{pd.unique(merged[['gene_a','gene_b']].values.ravel()).shape[0]:,}")


Saved merged_ppi.csv  - 232,221 edges, ['gene_a', 'gene_b', 'databases', 'n_databases']

Unique genes in merged network: 17,364


## Section 3 - Brain expression filter (Human Protein Atlas)

We use the **HPA brain-region RNA** dataset (`rna_brain_region_hpa.tsv`) and keep
genes whose nTPM exceeds `NTPM_THRESHOLD` (default 1) in **at least one of the 13
brain regions**. This file is brain-only (amygdala, basal ganglia, cerebellum,
cerebral cortex, choroid plexus, hippocampal formation, hypothalamus, medulla
oblongata, midbrain, pons, spinal cord, thalamus, white matter).

The merged PPI is then filtered to interactions where **both** partners are
brain-expressed, and saved twice: as a binary interaction list
(`data/PPI_binary.tsv`) and with metadata retained
(`dbs/merged_ppi_brain.csv`).

In [13]:
#  Human Protein Atlas brain-region RNA 
HPA_URL = "https://www.proteinatlas.org/download/tsv/rna_brain_region_hpa.tsv.zip"
HPA_TSV = data_dir / "rna_brain_region_hpa.tsv"

# Download + unzip on first run.
if not HPA_TSV.exists():
    import urllib.request, zipfile, io
    print(f"Downloading HPA brain regions RNA -> {HPA_TSV} ...")
    with urllib.request.urlopen(HPA_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(HPA_TSV.parent)
    print("done.")

# Expression threshold (nTPM)
NTPM_THRESHOLD = 1

hpa_brain = pd.read_csv(HPA_TSV, sep="\t")
BRAIN_REGIONS = sorted(hpa_brain["Brain region"].unique())
print(f"Brain regions used ({len(BRAIN_REGIONS)}): {BRAIN_REGIONS}")

brain_expr_all = (
    hpa_brain
    .pivot_table(index="Gene name", columns="Brain region", values="nTPM", aggfunc="max")
    .fillna(0.0)
)

brain_genes = set(
    brain_expr_all.index[(brain_expr_all > NTPM_THRESHOLD).any(axis=1)]
)

print(f"Genes in HPA brain table            : {len(brain_expr_all):,}")
print(f"Brain-expressed (nTPM > {NTPM_THRESHOLD} in any) : {len(brain_genes):,}")

# Save
with open(data_dir / "brain_genes.txt", "w") as f:
    f.write("\n".join(sorted(brain_genes)))

print(f"Saved to {data_dir}/brain_genes.txt")

Brain regions used (13): ['amygdala', 'basal ganglia', 'cerebellum', 'cerebral cortex', 'choroid plexus', 'hippocampal formation', 'hypothalamus', 'medulla oblongata', 'midbrain', 'pons', 'spinal cord', 'thalamus', 'white matter']
Genes in HPA brain table            : 20,151
Brain-expressed (nTPM > 1 in any) : 17,726
Saved to data/brain_genes.txt


In [14]:
merged_ppi = pd.read_csv(DB_DIR / "merged_ppi.csv")

ppi_binary = (
    merged_ppi[
        merged_ppi["gene_a"].isin(brain_genes) &
        merged_ppi["gene_b"].isin(brain_genes)
    ][["gene_a", "gene_b"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
ppi_binary.to_csv(data_dir / "PPI_binary.tsv", sep="\t", index=False)

n_genes = pd.unique(ppi_binary[["gene_a", "gene_b"]].values.ravel()).shape[0]
print(f"Brain-expressed binary PPI: {len(merged_ppi):,} -> {len(ppi_binary):,} "
      f"edges ({len(ppi_binary)/len(merged_ppi):.1%} kept), {n_genes:,} proteins")
print(f"Saved to {data_dir}/PPI_binary.tsv")

Brain-expressed binary PPI: 232,221 -> 197,413 edges (85.0% kept), 14,774 proteins
Saved to data/PPI_binary.tsv


## Section 4 - SCN1A-centred network (Dravet syndrome)

Dravet syndrome is caused in >80% of patients by loss-of-function variants in
**SCN1A** (Na<sub>v</sub>1.1). The network is therefore **seeded on SCN1A alone**
and grown outwards over the brain-filtered merged interactome.

**Expansion.** `MAX_HOPS` controls how far the neighbourhood grows from SCN1A:
hop 1 = SCN1A's direct interactors, hop 2 = the interactors of those interactors,
and so on. 

`MAX_HOPS = 2` is the default

In [15]:
merged_ppi = pd.read_csv(DB_DIR / "merged_ppi.csv")

merged_ppi_brain = merged_ppi[
    merged_ppi["gene_a"].isin(brain_genes) &
    merged_ppi["gene_b"].isin(brain_genes)
].copy()
merged_ppi_brain.to_csv(DB_DIR / "merged_ppi_brain.csv", index=False)

print(f"Brain-filtered PPI: {len(merged_ppi):,} -> {len(merged_ppi_brain):,} edges")
print(f"Saved to {DB_DIR}/merged_ppi_brain.csv")

Brain-filtered PPI: 232,221 -> 197,413 edges
Saved to dbs/merged_ppi_brain.csv


In [16]:
# Configuration 
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SEED_GENE = "SCN1A"       
MAX_HOPS  = 2             
NET_PREFIX = "scn1a"   

print(f"Seed gene: {SEED_GENE} | expansion: {MAX_HOPS} hop(s)")

Seed gene: SCN1A | expansion: 2 hop(s)


In [18]:
# Build the SCN1A neighbourhood
import networkx as nx

allowed_edges = pd.read_csv(DB_DIR / "merged_ppi_brain.csv")

G_brain = nx.from_pandas_edgelist(
    allowed_edges, "gene_a", "gene_b",
    edge_attr=["databases", "n_databases"],
)
G_brain.remove_edges_from(list(nx.selfloop_edges(G_brain)))
print(f"Brain interactome: {G_brain.number_of_nodes():,} proteins, "
      f"{G_brain.number_of_edges():,} interactions")

if SEED_GENE not in G_brain:
    raise ValueError(f"{SEED_GENE} is absent from the brain-filtered interactome")

hop_of = nx.single_source_shortest_path_length(G_brain, SEED_GENE, cutoff=MAX_HOPS + 1)

print(f"\nNeighbourhood growth around {SEED_GENE}:")
_cum = 0
for h in range(0, MAX_HOPS + 2):
    n_h = sum(1 for d in hop_of.values() if d == h)
    _cum += n_h
    tag = "  <- kept" if h == MAX_HOPS else ("  (not included)" if h > MAX_HOPS else "")
    print(f"  hop {h}: {n_h:>6,} new  |  cumulative {_cum:>6,}{tag}")

node_set = {g for g, d in hop_of.items() if d <= MAX_HOPS}
G = nx.Graph(G_brain.subgraph(node_set))

for n in G.nodes():
    G.nodes[n]["hop"]             = hop_of[n]
    G.nodes[n]["is_seed"]         = (n == SEED_GENE)
    G.nodes[n]["brain_expressed"] = n in brain_genes

n_iso = len(list(nx.isolates(G)))
print(f"\n{SEED_GENE} network ({MAX_HOPS} hops): {G.number_of_nodes():,} proteins, "
      f"{G.number_of_edges():,} interactions "
      f"({nx.number_connected_components(G)} component(s), {n_iso} isolated)")
print(f"{SEED_GENE} direct interactors: "
      f"{', '.join(sorted(G.neighbors(SEED_GENE)))}")

Brain interactome: 14,774 proteins, 197,413 interactions

Neighbourhood growth around SCN1A:
  hop 0:      1 new  |  cumulative      1
  hop 1:     14 new  |  cumulative     15
  hop 2:  1,006 new  |  cumulative  1,021  <- kept
  hop 3: 10,287 new  |  cumulative 11,308  (not included)

SCN1A network (2 hops): 1,021 proteins, 11,853 interactions (1 component(s), 0 isolated)
SCN1A direct interactors: CALM1, ESR1, ESR2, FGF11, FGF12, FGF13, FGF14, IL10, NEDD4L, PSEN1, SCN1B, SCN4B, SNTA1, TNF


### Export

The finished network is written to `results/scn1a_ppi_brain_network.graphml` and ready to open in
Cytoscape.

In [ ]:
# Export the SCN1A brain interactome
graphml_path = RESULTS_DIR / f"{NET_PREFIX}_ppi_brain_network.graphml"

nx.set_node_attributes(G, dict(G.degree()), "degree")
nx.write_graphml(G, graphml_path)

print(f"Saved {graphml_path}  -  {G.number_of_nodes():,} proteins, "
      f"{G.number_of_edges():,} interactions")